# GenomicPrediction.jl: 端到端完整示例

本笔记本演示了如何使用 `GenomicPrediction.jl` 包完成一个完整的基因组预测分析流程。

我们将涵盖以下步骤：
1.  **环境设置**: 激活项目环境并加载必要的包。
2.  **数据加载**: 从 CSV 文件加载基因型和表型数据。
3.  **数据预处理**: 对标记进行质量控制 (QC) 和缺失值填充。
4.  **模型训练**: 训练多种不同的预测模型 (GBLUP, BayesB)。
5.  **模型评估**: 使用交叉验证来比较模型的性能。
6.  **结果解释**: 提取并可视化 SNP 效应。

In [ ]:
import Pkg
# 激活当前目录下的项目环境
Pkg.activate(".")

# 加载包
using GenomicPrediction
using DataFrames
using Plots # 用于可视化

## 2. 数据加载

我们使用软件包测试目录中的示例数据。在实际应用中，您需要将路径替换为您的数据文件路径。

In [ ]:
geno_path = "../../test/sample_data/genotypes.csv"
pheno_path = "../../test/sample_data/phenotypes.csv"

original_data = load_csv(geno_path, pheno_path)

## 3. 数据预处理

我们首先对数据进行质量控制，过滤掉低质量的标记，然后用平均值填充剩余的缺失数据（如果存在）。

In [ ]:
# 过滤 MAF 低于 0.05 或调用率低于 0.90 的标记
qc_data = filter_markers(original_data; maf_threshold=0.05, call_rate_threshold=0.90)

# 对任何剩余的缺失值进行平均值填充
# 注意：我们的示例数据没有缺失值，但这演示了如何调用该函数
imputed_data = impute_mean(qc_data)

## 4. 模型训练

我们将训练两个模型：GBLUP 和 BayesB，以便后续进行比较。

In [ ]:
# 初始化模型
gblup_model = GBLUPModel(lambda=50.0)
bayesb_model = BayesBModel(iterations=2000, burnin=500, pi=0.05)

# 训练模型 (使用非原地修改的 `fit` 版本)
println("--- 训练 GBLUP 模型 ---")
trained_gblup = fit(gblup_model, imputed_data)

println("\n--- 训练 BayesB 模型 ---")
trained_bayesb = fit(bayesb_model, imputed_data)

## 5. 模型评估

使用 5-折交叉验证来评估和比较两个模型的预测准确性。

In [ ]:
println("--- GBLUP 交叉验证 ---")
cv_results_gblup = cross_validate(GBLUPModel(lambda=50.0), imputed_data, 5)
println("GBLUP 平均准确率: ", cv_results_gblup.metrics["mean_accuracy"])

println("\n--- BayesB 交叉验证 ---")
cv_results_bayesb = cross_validate(BayesBModel(iterations=1000, burnin=200), imputed_data, 5)
println("BayesB 平均准确率: ", cv_results_bayesb.metrics["mean_accuracy"])

## 6. 结果解释

我们可以从训练好的模型中提取 SNP 效应，并用条形图将其可视化。

In [ ]:
snp_effects_gblup = get_snp_effects(trained_gblup, imputed_data)
snp_effects_bayesb = get_snp_effects(trained_bayesb, imputed_data)

println("GBLUP SNP 效应:")
println(first(snp_effects_gblup, 5))

println("BayesB SNP 效应:")
println(first(snp_effects_bayesb, 5))

In [ ]:
# 使用 Plots.jl 可视化效应
p1 = bar(snp_effects_gblup.MarkerName, snp_effects_gblup.Effect, 
         title="GBLUP SNP Effects", legend=false, xrotation=45)

p2 = bar(snp_effects_bayesb.MarkerName, snp_effects_bayesb.Effect, 
         title="BayesB SNP Effects", legend=false, xrotation=45)

plot(p1, p2, layout=(2, 1), size=(800, 600))